# Teach: grab the electrode standing upright at sample 0 (for the dip with rotation)

For the dip with rotation, the gripper points straight down and holds the electrode by the **top of its metal
part**, so the electrode hangs straight down and turning the gripper spins it in place. This notebook finds that
grab and checks it in the air. **No ink, no imaging rack.**

- **Cell 1**: settings. Nothing moves.
- **Cell 2**: TEACH the grab. The arm goes high, across to above sample 0, opens the gripper and comes down to
  `START_Z`. You lower it with typed commands, grab, lift the electrode out of its stand, and type `done`.
  It prints a `Grab:` line for Claude.
- **Cell 3**: CHECK in the air, still holding it, high above mid_station:
  - a) a half turn and back: does the electrode spin in place?
  - b) the quarter turn of the hand that will lay it flat for the imaging rack, coated end to the right: is it
    face up and level? (`flip` tries the other way.) It prints a `Face up:` line for Claude.
- **Cell 4**: PUT BACK. Lowers the electrode back into its stand at sample 0, lets go slowly, and goes up.

Before running: the electrode standing upright at sample 0, metal part up, dry; gripper empty; hand on the e-stop.

In [ ]:
from robotic_testing.platforms import init_vla_platform

rt = init_vla_platform(exp_name='rotation_grab_teach', connect_db=False)
arm = rt.arm

In [ ]:
# ===== 1/4: settings. Nothing moves in this cell. =====
import time

SAMPLE = 0                    # the electrode stands upright at this sample's place (same x and y), higher up
SAFE_Z = 500.0                # travel height, high above the bench
START_Z = 400.0               # the open gripper stops here above the electrode, for you to lower it from. It must
                              # be clearly above the top of the electrode; if it is not, make this bigger (at most 480).
LOWEST_Z = 200.0              # while teaching, the gripper never goes lower than this
AIR_SPOT = [347.9, 308.1, 500.0]   # where cell 3 does its checks: high above mid_station, in open air
CHECK_TURN_DEG = 180.0        # cell 3 turns the electrode this far and back, to see whether it spins in place
CHECK_TURN_SPEED = 30.0       # degrees per second for that turn
RELEASE_STEP = 1              # cell 4 lets go slowly, as on the imaging rack: this many units at a time,
RELEASE_STEP_PAUSE_S = 0.15   #   pausing this long after each step

if not LOWEST_Z + 20 <= START_Z <= 480:
    raise SystemExit(f'START_Z must be between {LOWEST_Z + 20:g} and 480.')

cfg = arm.config
_sample = [float(v) for v in cfg.pos_cal.get_sample_coord(SAMPLE)]
SX, SY = _sample[0], _sample[1]       # x, y of the sample's place
GRAB_YAW = _sample[5]                 # gripper turned as for the usual pick-up (you can turn it while teaching)
DOWN = [180.0, 0.0]                   # roll, pitch with the gripper pointing straight down
J7_LIMIT = 350.0                      # the last joint turns between -360 and +360 degrees; stay 10 inside that
# Laying the electrode flat, coated end to the right, as on the imaging rack: first the gripper turns so its
# fingers close front-and-back (yaw 180 or 0), then the hand turns a quarter turn about the line between the
# fingers, so the gripper points to the right and the electrode sticks out to the right of it. Which of the two
# ends with the face up depends on how the electrode sits in the fingers; cell 3 tries one, and flip tries the other.
FLAT = {180.0: [0.0, 90.0, 0.0], 0.0: [180.0, -90.0, 0.0]}   # yaw before -> roll, pitch, yaw when flat


def at_speed(mm_s):
    """The arm's default motion settings with the speed swapped to mm_s."""
    return {**cfg.pos_settings_dict['default'], 'speed': mm_s}


def wrap(angle):
    """An angle in degrees, brought into -180 .. 180."""
    return (angle + 180.0) % 360.0 - 180.0


def pointing_down(pose, tol=5.0):
    return abs(wrap(pose[3] - 180.0)) <= tol and abs(pose[4]) <= tol


def j7():
    """Angle of the last joint (the one that turns the gripper), in degrees."""
    return arm.get_joint_angles()[6]


def turn_j7_to(angle, speed):
    """Turn only the last joint, to angle (degrees), at speed (degrees per second). Nothing else moves."""
    if abs(angle) > J7_LIMIT:
        raise SystemExit(f'That would turn the last joint to {angle:.0f} degrees, past its range. Stopped; tell Claude.')
    arm.set_servo_angle(servo_id=7, angle=angle, speed=speed, mvacc=200, wait=True)


def learn_turn_direction():
    """
    With the gripper pointing straight down, turn the last joint 5 degrees (towards the middle of its range) and
    back, to learn which way that turns the gripper.
    """
    global YAW_PER_J7
    a, yaw0 = j7(), arm.get_cartesian_pos()[5]
    step = -5.0 if a > 0 else 5.0
    turn_j7_to(a + step, 10)
    yaw1 = arm.get_cartesian_pos()[5]
    turn_j7_to(a, 10)
    ratio = wrap(yaw1 - yaw0) / step
    if abs(abs(ratio) - 1) > 0.3:
        raise SystemExit(f'Turning the last joint {step:+g} degrees turned the gripper {wrap(yaw1 - yaw0):+.1f} degrees; '
                         f'expected about 5. Not safe to continue. Tell Claude.')
    YAW_PER_J7 = 1.0 if ratio > 0 else -1.0


def turn_gripper_to(yaw, speed=10, centre=False):
    """
    With the gripper pointing straight down, turn it about its own axis to yaw, with the last joint only (so it
    stays exactly where it is). If the short way would take the last joint past its range, it goes the long way.
    centre=True: of the ways round, take the one that leaves the last joint nearest the middle of its range.
    """
    if YAW_PER_J7 is None:
        learn_turn_direction()
    a = j7()
    target = a + wrap(yaw - arm.get_cartesian_pos()[5]) / YAW_PER_J7
    choices = [t for t in (target, target - 360, target + 360) if abs(t) <= J7_LIMIT]
    best = min(choices, key=abs) if centre else min(choices, key=lambda t: abs(t - a))
    if abs(best - a) > 180:
        print('   (turning the long way round, ' + ('so the last joint has room for what comes next)' if centre
              else 'to stay inside the last joint\'s range)'))
    turn_j7_to(best, speed)


def ask(question, answers):
    """Ask until one of the answers is typed; return it."""
    while True:
        answer = input(question).strip().lower()
        if answer in answers:
            return answer
        print('Type ' + ' or '.join(answers) + '.')


def let_go():
    """Open the fingers one small step at a time, with a pause after each, all the way open; then let it settle."""
    fully_open = int(cfg.gripper_open_dist)
    pos = int(round(arm.gripper_opening))         # about 9 when the fingers hold an electrode
    while pos < fully_open:
        pos = min(pos + int(RELEASE_STEP), fully_open)
        arm.set_gripper_position(pos, wait=True, speed=1000, auto_enable=True)
        time.sleep(RELEASE_STEP_PAUSE_S)
    time.sleep(0.5)


# remembered across cells (and kept if this cell is run again)
holding = globals().get('holding', False)   # does the gripper hold the electrode?
GRAB = globals().get('GRAB')                # set by cell 2: the pose at which the gripper holds it in its stand
CLEAR_Z = globals().get('CLEAR_Z')          # set by cell 2: the height at which it is completely out of its stand
YAW_PER_J7 = globals().get('YAW_PER_J7')    # +1 or -1: how the gripper turns when the last joint turns (learned)

print(f'Settings loaded: sample {SAMPLE} is at x = {SX:.1f}, y = {SY:.1f}; the gripper will start at z = {START_Z:g} '
      f'above it. Nothing has moved.')

In [ ]:
# ===== 2/4: TEACH the grab. Gripper EMPTY; the electrode standing upright at sample 0, metal part up. =====
# The arm goes straight up, across to above sample 0, opens the gripper and comes down to START_Z. Then you move it
# with typed commands (directions as seen standing in front of the bench, facing the wall):
#   d 10 / u 10   -> 10 mm lower / higher (at most 20 mm at a time; use 1-2 mm when close)
#   l 2 / r 2     -> 2 mm left / right      f 2 / b 2 -> 2 mm towards you / towards the wall (at most 5 mm)
#   t 10 / t -10  -> turn the gripper 10 degrees clockwise / counterclockwise, seen from above (at most 90)
#   grab          -> close the fingers here
#   open          -> open the fingers again (only where it grabbed, so nothing falls)
#   done          -> after grab, and after lifting the electrode out of its stand with u: go straight up,
#                    holding it, and print the grab for Claude
#   back          -> give up (fingers open): go straight up
# Aim: the fingers come down around the TOP of the metal part and close on its two narrow sides (the same sides
# they hold today), with the metal part in the middle between the fingers, as deep as the metal part allows
# without touching the electrode itself. Turn (t) while the fingers are still above the metal part, until they
# line up with those sides. Once it is grabbed, only u and d work (no sideways moves, so the stand is not pulled):
# lift with u until the electrode is completely out of its stand, then type done.
here = arm.get_cartesian_pos()
if not pointing_down(here) or here[2] < 240:
    raise SystemExit('The gripper must point straight down and be at least 5 cm above the racks (z >= 240) '
                     'before this cell. The arm did not move.')
if holding:
    raise SystemExit('The gripper still holds the electrode. Run cell 4 to put it back first. The arm did not move.')
opening = arm.gripper_opening
if 3 <= opening < 45:
    raise SystemExit(f'The gripper seems to be holding something (opening {opening:.1f}). It must be empty for '
                     f'this cell. The arm did not move.')
print(f'TEACH the grab at sample {SAMPLE} (x = {SX:.1f}, y = {SY:.1f}). The arm will go straight up to z = {SAFE_Z:g},')
print(f'across to above sample {SAMPLE}, open the gripper, and come down to z = {START_Z:g}.')
if input(f'Gripper empty, electrode standing upright at sample {SAMPLE} (metal part up), nothing else under the '
         f'gripper, hand on the e-stop? Type yes to start: ').strip().lower() != 'yes':
    raise SystemExit('Cancelled, the arm did not move.')
arm.assert_ready()
print('-> straight up')
arm.move_to_pos([here[0], here[1], SAFE_Z, *here[3:]], **at_speed(30))
print(f'-> across, to above sample {SAMPLE}')
arm.move_to_pos([SX, SY, SAFE_Z, *here[3:]], **at_speed(30))
learn_turn_direction()
turn_gripper_to(GRAB_YAW, speed=30)
arm.open_gripper()
print(f'-> down to z = {START_Z:g}')
arm.move_to_pos([SX, SY, START_Z, *DOWN, GRAB_YAW], **at_speed(20))

MOVES = {'l': (-1, 0), 'r': (1, 0), 'f': (0, -1), 'b': (0, 1)}
tx, ty, tz, tyaw = SX, SY, START_Z, GRAB_YAW
turned = 0.0              # degrees turned clockwise (seen from above) from the usual pick-up
grab_at = None
print('Ready. Lower the gripper (d) until the fingers are around the top of the metal part, then type grab.')
while True:
    cmd = input('d/u, l/r/f/b + mm, t + degrees, or grab, open, done, back: ').strip().lower().split()
    if not cmd:
        continue
    word = cmd[0]
    if cmd == ['grab']:
        if holding:
            print('Already holding it. Lift it out of its stand with u, then type done.')
            continue
        arm.close_gripper()
        opening = arm.gripper_opening
        if opening < 3:
            print(f'The fingers closed on nothing (opening {opening:.1f}). Opening again; move and try again.')
            arm.open_gripper()
            continue
        holding = True
        grab_at = [tx, ty, tz, *DOWN, tyaw]
        print(f'Holding it (opening {opening:.1f}). Lift it out of its stand with u (e.g. u 5), then type done.')
        continue
    if cmd == ['open']:
        if holding and tz > grab_at[2] + 0.5:
            print('It would fall. Lower it back into its stand first (d), then type open.')
            continue
        arm.open_gripper()
        holding, grab_at = False, None
        print('Fingers open.')
        continue
    if cmd == ['done']:
        if not holding:
            print('Grab it first.')
            continue
        if ask('Is the electrode completely out of its stand, with nothing under or beside it? Type yes to go '
               'straight up, or no to keep adjusting: ', ('yes', 'no')) == 'yes':
            break
        continue
    if cmd == ['back']:
        if holding:
            print('Still holding it. Lower it back into its stand (d), type open, then back.')
            continue
        break
    if len(cmd) != 2 or word not in ('d', 'u', 'l', 'r', 'f', 'b', 't'):
        print('Did not understand. Examples: d 10, d 1, u 5, l 2, f 1, t 10, t -5, grab, open, done, back.')
        continue
    try:
        n = float(cmd[1])
    except ValueError:
        print('The second part must be a number, e.g. d 5.')
        continue
    nx, ny, nz, nyaw = tx, ty, tz, tyaw
    if word in ('d', 'u'):
        if not 0 < n <= 20:
            print('Use a distance between 0 and 20 mm.')
            continue
        nz = tz - n if word == 'd' else tz + n
        if not LOWEST_Z <= nz <= SAFE_Z:
            print(f'That is outside {LOWEST_Z:g} - {SAFE_Z:g} mm in height, so the arm did not move.')
            continue
        if holding and nz < grab_at[2] - 0.01:
            print('That would push the electrode down into its stand, so the arm did not move.')
            continue
        speed = 5 if holding else 10
    elif word == 't':
        if holding:
            print('Not while holding it. (To turn the grab: lower it back, open, lift a little, then turn.)')
            continue
        if not 0 < abs(n) <= 90:
            print('Use an angle between -90 and 90 degrees (not 0).')
            continue
        if abs(turned + n) > 180:
            print('That would turn the gripper more than half a turn from the usual pick-up, so it did not move.')
            continue
        turned += n
        tyaw = wrap(tyaw - n)           # clockwise seen from above = yaw going down
        turn_gripper_to(tyaw)
        print(f'Turned. Now turned {turned:+g} degrees from the usual pick-up.')
        continue
    else:
        if holding:
            print('Not while holding it (it would pull on the stand). Lift it out with u, then type done.')
            continue
        if not 0 < n <= 5:
            print('Use a distance between 0 and 5 mm.')
            continue
        dx, dy = MOVES[word]
        nx, ny = tx + dx * n, ty + dy * n
        if max(abs(nx - SX), abs(ny - SY)) > 30:
            print(f'That would be more than 30 mm from sample {SAMPLE}, so the arm did not move.')
            continue
        speed = 5
    tx, ty, tz, tyaw = nx, ny, nz, nyaw
    arm.move_to_pos([tx, ty, tz, *DOWN, tyaw], **at_speed(speed))
    print(f'Moved. Now x = {tx:.1f}, y = {ty:.1f}, z = {tz:.1f}, gripper turned {turned:+g} degrees from the usual pick-up'
          + (f'; {tz - grab_at[2]:.1f} mm above where it grabbed.' if holding else '.'))

print('-> straight up')
arm.move_to_pos([tx, ty, SAFE_Z, *DOWN, tyaw], **at_speed(20))
if holding:
    GRAB, CLEAR_Z = grab_at, tz
    print(f'Grab: x = {GRAB[0]:.1f}, y = {GRAB[1]:.1f}, z = {GRAB[2]:.1f}, yaw = {GRAB[5]:.1f}; '
          f'out of its stand at z = {CLEAR_Z:.1f}   <- send this line to Claude')
    print('Holding the electrode high above its stand. Next: cell 3 (checks in the air), or cell 4 to put it back.')
else:
    print('Nothing grabbed; the gripper is open and high above the bench.')

In [ ]:
# ===== 3/4: CHECK in the air, holding the electrode (after cell 2). Nothing touches anything. =====
# High above mid_station:
#   a) a half turn and back: does the electrode hang straight and spin in place, or does its tip swing round?
#   b) the quarter turn of the hand that will lay it flat on the imaging rack, coated end to the right: is it
#      face up and level? If it comes out face down, type flip to try the other way.
# Then it turns back to pointing straight down, still holding the electrode.
if not holding or GRAB is None:
    raise SystemExit('The gripper is not holding the electrode. Run cell 2 first. The arm did not move.')
here = arm.get_cartesian_pos()
if not pointing_down(here) or here[2] < SAFE_Z - 5:
    raise SystemExit(f'The gripper must point straight down at z = {SAFE_Z:g}, where cell 2 leaves it. '
                     f'The arm did not move.')
if input('Holding the electrode, nothing near the space high above mid_station, hand on the e-stop? '
         'Type yes to start: ').strip().lower() != 'yes':
    raise SystemExit('Cancelled, the arm did not move.')
arm.assert_ready()
print('-> across, to high above mid_station')
arm.move_to_pos([*AIR_SPOT, *here[3:]], **at_speed(30))

print(f'a) turning the electrode {CHECK_TURN_DEG:g} degrees and back, at {CHECK_TURN_SPEED:g} deg/s: watch its bottom tip')
a = j7()
turn_j7_to(a - CHECK_TURN_DEG if a > 0 else a + CHECK_TURN_DEG, CHECK_TURN_SPEED)
turn_j7_to(a, CHECK_TURN_SPEED)
spins = ask('Did it hang straight down and spin in place (its tip did not swing round in a circle)? yes / no: ',
            ('yes', 'no'))

pre, face = 180.0, None
for attempt in range(3):
    print('b) turning the fingers front-and-back, then laying the electrode flat, coated end to the right')
    turn_gripper_to(pre, speed=30, centre=True)
    arm.move_to_pos([*AIR_SPOT, *FLAT[pre]], **at_speed(20))
    answer = ask('It should be lying flat, pointing to the right. Type yes (face UP and level), flip (face DOWN), '
                 'edge (on its edge), or no (something else is wrong): ', ('yes', 'flip', 'edge', 'no'))
    print('   back to pointing straight down')
    arm.move_to_pos([*AIR_SPOT, *DOWN, pre], **at_speed(20))
    if answer != 'flip':
        face = answer
        break
    pre = 0.0 if pre == 180.0 else 180.0
turn_gripper_to(GRAB[5], speed=30)

if face == 'yes':
    print(f'Face up: yaw {pre:g} before laying it flat, flat at roll/pitch/yaw {FLAT[pre]}; '
          f'spins in place: {spins}   <- send this line to Claude')
elif face == 'edge':
    print('It lay on its edge: the fingers hold the metal part by its flat faces, not its narrow sides. Put it back '
          '(cell 4) and teach the grab again (cell 2), turning the gripper a quarter turn (t 90) before grabbing.')
else:
    print(f'Not right yet (spins in place: {spins}). Tell Claude what you saw.')
print('Pointing straight down again, holding the electrode. Next: cell 4 puts it back in its stand.')

In [ ]:
# ===== 4/4: PUT BACK the electrode in its stand at sample 0, let go slowly, and go up =====
if not holding or GRAB is None:
    raise SystemExit('The gripper is not holding the electrode, so there is nothing to put back. The arm did not move.')
here = arm.get_cartesian_pos()
lying_flat_in_air = (max(abs(a - b) for a, b in zip(here[:3], AIR_SPOT)) <= 5 and abs(abs(here[4]) - 90) <= 5)
if not pointing_down(here) and not lying_flat_in_air:
    raise SystemExit('The gripper is not pointing straight down. Tell Claude. The arm did not move.')
if here[2] < 240:
    raise SystemExit('The gripper is lower than z = 240. Tell Claude. The arm did not move.')
if input(f'Stand at sample {SAMPLE} empty and in place, hand on the e-stop? Type yes to put the electrode back: '
         ).strip().lower() != 'yes':
    raise SystemExit('Cancelled, the arm did not move.')
arm.assert_ready()
if lying_flat_in_air:
    print('-> back to pointing straight down')
    arm.move_to_pos([*AIR_SPOT, *DOWN, 180.0 if here[4] > 0 else 0.0], **at_speed(20))
    here = arm.get_cartesian_pos()
print(f'-> up and across, high above the bench, to above sample {SAMPLE}')
arm.move_to_pos([here[0], here[1], SAFE_Z, *here[3:]], **at_speed(30))
arm.move_to_pos([GRAB[0], GRAB[1], SAFE_Z, *here[3:]], **at_speed(30))
turn_gripper_to(GRAB[5], speed=30)
print(f'-> down to z = {CLEAR_Z:.1f}, just above its stand')
arm.move_to_pos([GRAB[0], GRAB[1], CLEAR_Z, *GRAB[3:]], **at_speed(20))
if ask('Electrode right above its stand, lined up to go in? Type yes to lower it in, or no to stop here: ',
       ('yes', 'no')) == 'no':
    raise SystemExit('Stopped above the stand, still holding the electrode. Run this cell again when ready.')
print(f'-> lowering it into its stand at 3 mm/s, to z = {GRAB[2]:.1f}')
arm.move_to_pos(GRAB, **at_speed(3))
print('-> letting go, slowly')
let_go()
holding = False
print('-> up, off the metal part, then high above the bench')
arm.move_to_pos([GRAB[0], GRAB[1], CLEAR_Z, *GRAB[3:]], **at_speed(5))
arm.move_to_pos([GRAB[0], GRAB[1], SAFE_Z, *GRAB[3:]], **at_speed(30))
print(f'Done. The electrode is back in its stand at sample {SAMPLE}; the gripper is open, high above it.')